In [16]:
import pandas as pd
import os
import geopandas as gpd
import glob


In [17]:
GTFS_ROUTE_TYPES = {
    "Bus": 3,
    "Autobús": 3,
    "Micro": 3,
    "Combi": 3,
    "Colectivo": 3,
    "Metro": 1,
    "Subway": 1,
    "Tren Ligero": 0,
    "Tram": 0,
    "Tren": 2,
    "Rail": 2,
    "Ferry": 4,
    "Barco": 4,
    "Teleférico": 6,
    "Cablebús": 6,
    "Trolebús": 11,
    "Funicular": 7
}

In [32]:


geojson_paths = glob.glob("./data/proc/*.geojson")

for geojson_path in geojson_paths:
    print(geojson_path)

    ### Read shapes from raw geojson
    path_routes_raw = geojson_path
    ## Read shapes from raw geojson
    routes_raw = gpd.read_file(path_routes_raw)
    print(f"Raw shapes (# routes): {routes_raw.shape}")
    routes_raw.head()


    routes_clean = routes_raw.copy()
    routes_clean = routes_clean[["route_name", "route_name_short", "type", "geometry"]]
    routes_clean.head()


    duplicados_count = routes_clean["route_name"].duplicated().sum()
    print(f"Total de route_name duplicados: {duplicados_count}")

    # keep=False muestra todas las ocurrencias del duplicado, no solo la segunda
    df_duplicados = routes_clean[routes_clean.duplicated(subset=["route_name"], keep=False)]
    df_duplicados.sort_values("route_name")

    # 1. Identify which rows are duplicated (all occurrences)
    es_duplicado = routes_clean.duplicated(subset=['route_name'], keep=False)

    # 2. Generate suffix (0 -> a, 1 -> b, etc.) using ASCII code
    # cumcount() gives 0 for the first time it sees an ID, 1 for the second, etc.
    sufijos = routes_clean.groupby('route_name').cumcount().map(lambda x: f"_{chr(97 + x)}")

    # 3. Apply change only where there are duplicates
    routes_clean.loc[es_duplicado, 'route_name'] = routes_clean.loc[es_duplicado, 'route_name'].astype(str) + sufijos


    routes_clean["route_type"] = routes_clean["type"].map(GTFS_ROUTE_TYPES).fillna(3).astype(int)

    print(routes_clean.columns)

    routes_clean = routes_clean[[ "route_name", "route_name_short",  "route_type", "geometry"]].copy()
    routes_clean.head(2)


    # Reproject
    print(routes_clean.crs)  # should say EPSG:4326
    routes_clean = routes_clean.to_crs(epsg=32614) # Reproject to UTM 14N (meters)

    # Calculate length in meters
    routes_clean["length_m"] = routes_clean.geometry.length
    routes_clean.sort_values("length_m")

    # Remove rows where length_m is NaN
    routes_clean = routes_clean.dropna(subset=["length_m"])


    routes_clean.drop(columns=["length_m"], inplace=True)
    routes_clean.head()

    # save in a unique path based on geojson_path
    path_routes_clean = geojson_path.replace("proc", "export")
    routes_clean.to_file(path_routes_clean)



./data/proc/mazatlán.geojson
Raw shapes (# routes): (12, 6)
Total de route_name duplicados: 0
Index(['route_name', 'route_name_short', 'type', 'geometry', 'route_type'], dtype='object')
EPSG:4326
./data/proc/culiacán.geojson
Raw shapes (# routes): (69, 6)
Total de route_name duplicados: 1
Index(['route_name', 'route_name_short', 'type', 'geometry', 'route_type'], dtype='object')
EPSG:4326
./data/proc/morelia.geojson
Raw shapes (# routes): (63, 6)
Total de route_name duplicados: 3
Index(['route_name', 'route_name_short', 'type', 'geometry', 'route_type'], dtype='object')
EPSG:4326
./data/proc/torreón.geojson
Raw shapes (# routes): (29, 6)
Total de route_name duplicados: 0
Index(['route_name', 'route_name_short', 'type', 'geometry', 'route_type'], dtype='object')
EPSG:4326
./data/proc/reynosa.geojson
Raw shapes (# routes): (39, 6)
Total de route_name duplicados: 0
Index(['route_name', 'route_name_short', 'type', 'geometry', 'route_type'], dtype='object')
EPSG:4326
./data/proc/puebla.geoj

/Users/danielbustillos/miniconda3/envs/analisis-general/lib/python3.10/site-packages/pyogrio/raw.py:723: RuntimeWarning: Infinite or NaN coordinate encountered
  ogr_write(


Raw shapes (# routes): (259, 6)
Total de route_name duplicados: 2
Index(['route_name', 'route_name_short', 'type', 'geometry', 'route_type'], dtype='object')
EPSG:4326
./data/proc/xalapa_de_enríquez.geojson
Raw shapes (# routes): (63, 6)
Total de route_name duplicados: 0
Index(['route_name', 'route_name_short', 'type', 'geometry', 'route_type'], dtype='object')
EPSG:4326
./data/proc/cancún.geojson
Raw shapes (# routes): (36, 6)
Total de route_name duplicados: 0
Index(['route_name', 'route_name_short', 'type', 'geometry', 'route_type'], dtype='object')
EPSG:4326
./data/proc/saltillo.geojson
Raw shapes (# routes): (59, 6)
Total de route_name duplicados: 2
Index(['route_name', 'route_name_short', 'type', 'geometry', 'route_type'], dtype='object')
EPSG:4326
./data/proc/hermosillo.geojson
Raw shapes (# routes): (33, 6)
Total de route_name duplicados: 0
Index(['route_name', 'route_name_short', 'type', 'geometry', 'route_type'], dtype='object')
EPSG:4326
./data/proc/ciudad_de_méxico.geojson
R

/Users/danielbustillos/miniconda3/envs/analisis-general/lib/python3.10/site-packages/shapely/measurement.py:182: RuntimeWarning: invalid value encountered in length
  return lib.length(geometry, **kwargs)


Raw shapes (# routes): (403, 6)
Total de route_name duplicados: 1
Index(['route_name', 'route_name_short', 'type', 'geometry', 'route_type'], dtype='object')
EPSG:4326
./data/proc/guadalajara.geojson
Raw shapes (# routes): (258, 6)
Total de route_name duplicados: 0
Index(['route_name', 'route_name_short', 'type', 'geometry', 'route_type'], dtype='object')
EPSG:4326
./data/proc/los_mochis.geojson
Raw shapes (# routes): (27, 6)
Total de route_name duplicados: 0
Index(['route_name', 'route_name_short', 'type', 'geometry', 'route_type'], dtype='object')
EPSG:4326
./data/proc/san_luis_potosí.geojson
Raw shapes (# routes): (58, 6)
Total de route_name duplicados: 0
Index(['route_name', 'route_name_short', 'type', 'geometry', 'route_type'], dtype='object')
EPSG:4326
./data/proc/mérida.geojson
Raw shapes (# routes): (56, 6)
Total de route_name duplicados: 0
Index(['route_name', 'route_name_short', 'type', 'geometry', 'route_type'], dtype='object')
EPSG:4326
./data/proc/colima.geojson
Raw shapes

Raw shapes (# routes): (11, 6)


,city,route_name,route_name_short,type,shape_id,geometry
0,Tijuana,A_Santa Fe - Pórticos - Línea,Santa Fe - Pórticos - Línea,Bus,shape_2007,"LINESTRING (-117.06399 32.43873, -117.06346 32..."
1,Tijuana,B_Otay-Mirador,Otay-Mirador,Bus,shape_2004,"LINESTRING (-117.07842 32.51154, -117.08008 32..."
2,Tijuana,C_Soler Libertad 70 76,Soler Libertad 70 76,Bus,shape_2005,"LINESTRING (-117.0808 32.52297, -117.08027 32...."
3,Tijuana,D_Central Camionera - Hospital - Soler Playas,Central Camionera - Hospital - Soler Playas,Bus,shape_2006,"LINESTRING (-116.94723 32.50531, -116.94728 32..."
4,Tijuana,G_Corredor 2000-Otay-Centro,Corredor 2000-Otay-Centro,Bus,shape_1999,"LINESTRING (-117.02916 32.53901, -117.029 32.5..."


### Format and clean routes

From `routes_raw` and notebook parameters, create the table with columns: `route_id | agency_id | route_name_short | route_long_name | route_type`.

,route_name,route_name_short,type,geometry
0,A_Santa Fe - Pórticos - Línea,Santa Fe - Pórticos - Línea,Bus,"LINESTRING (-117.06399 32.43873, -117.06346 32..."
1,B_Otay-Mirador,Otay-Mirador,Bus,"LINESTRING (-117.07842 32.51154, -117.08008 32..."
2,C_Soler Libertad 70 76,Soler Libertad 70 76,Bus,"LINESTRING (-117.0808 32.52297, -117.08027 32...."
3,D_Central Camionera - Hospital - Soler Playas,Central Camionera - Hospital - Soler Playas,Bus,"LINESTRING (-116.94723 32.50531, -116.94728 32..."
4,G_Corredor 2000-Otay-Centro,Corredor 2000-Otay-Centro,Bus,"LINESTRING (-117.02916 32.53901, -117.029 32.5..."


### Fix duplicates
Verify that there are no duplicate route ids and if there are duplicates, add subindex _a, _b, _etc

Total de route_name duplicados: 0


,route_name,route_name_short,type,geometry


Format columns

In [ ]:
#routes_clean.rename(columns={"route_name_short": "route_long_name", 
 #                               "type": "route_type"
  #                              }, inplace=True)

#routes_clean["route_name_short"] =  routes_clean["route_name"].astype(str)
# routes_clean["shape_id"] = "Shape_" + routes_clean["route_name"]





Index(['route_name', 'route_name_short', 'type', 'geometry', 'route_type'], dtype='object')


,route_name,route_name_short,route_type,geometry
0,A_Santa Fe - Pórticos - Línea,Santa Fe - Pórticos - Línea,3,"LINESTRING (-117.06399 32.43873, -117.06346 32..."
1,B_Otay-Mirador,Otay-Mirador,3,"LINESTRING (-117.07842 32.51154, -117.08008 32..."


Remove nans in geometry

EPSG:4326


,route_name,route_name_short,route_type,geometry
0,A_Santa Fe - Pórticos - Línea,Santa Fe - Pórticos - Línea,3,"LINESTRING (-1209939.195 3736657.443, -1209888..."
1,B_Otay-Mirador,Otay-Mirador,3,"LINESTRING (-1209879.361 3745138.289, -1210039..."
2,C_Soler Libertad 70 76,Soler Libertad 70 76,3,"LINESTRING (-1209880.489 3746471.562, -1209829..."
3,D_Central Camionera - Hospital - Soler Playas,Central Camionera - Hospital - Soler Playas,3,"LINESTRING (-1197426.787 3742235.081, -1197431..."
4,G_Corredor 2000-Otay-Centro,Corredor 2000-Otay-Centro,3,"LINESTRING (-1204613.06 3747418.344, -1204598...."
